In [2]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset
from torchvision import datasets, transforms
from torchvision.models.resnet import ResNet, BasicBlock
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader, Subset
import torch.nn.functional as F
from PIL import Image
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [ ]:
import torch
from torchvision import datasets, transforms

# Define image transformations
transform = transforms.Compose([
    transforms.Resize((32, 32)),         # resize images to 32x32
    transforms.ToTensor(),               # convert image to tensor
    transforms.Normalize((0.5,), (0.5,)) # normalize pixel values
])

# Load CIFAR-10 dataset
train_data = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_data = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)


28.6%

In [ ]:
labels = {
    0: "airplane",
    1: "automobile",
    2: "bird",
    3: "cat",
    4: "deer",
    5: "dog",
    6: "frog",
    7: "horse",
    8: "ship",
    9: "truck"
}

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
import numpy as np

# Transform
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Load CIFAR-10
train_data = datasets.CIFAR10(root="./data", train=True, download=True)

# Filter for cat and dog
cat_label = 3
dog_label = 5
targets_array = np.array(train_data.targets)
cat_dog_indices = np.where((targets_array == cat_label) | (targets_array == dog_label))[0]
filtered_images = train_data.data[cat_dog_indices]
filtered_labels = np.array([0 if label==cat_label else 1 for label in targets_array[cat_dog_indices]])

# Custom Dataset
class CatDogDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform
    def __len__(self):
        return len(self.images)
    def __getitem__(self, idx):
        img = self.images[idx]
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

# Dataset & DataLoader
cat_dog_train = CatDogDataset(filtered_images, filtered_labels, transform=transform)
train_loader = DataLoader(cat_dog_train, batch_size=32, shuffle=True)

# Test batch
images, labels = next(iter(train_loader))
print(f"Batch shape: {images.shape}, Labels: {labels[:10]}")



In [ ]:
from torch.utils.data import Dataset

class CatDogDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]

        # Convert to PIL Image before applying transform
        from PIL import Image
        image = Image.fromarray(image)

        if self.transform:
            image = self.transform(image)
        
        return image, label


In [ ]:
# Create CatDogDataset with both images and labels
cat_dog_train = CatDogDataset(filtered_images, filtered_labels, transform=transform)

# Print total number of images
print("Total images:", len(cat_dog_train))


In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(cat_dog_train, batch_size=32, shuffle=True)
